In [25]:
import anthropic
import pathlib
import csv
import pandas as pd
from dotenv import load_dotenv

In [26]:
load_dotenv()  # loads .env into environment variables

True

In [27]:
client = anthropic.Anthropic()

In [4]:
import os

In [7]:
os.chdir("csv/QUIMICA")

In [8]:
df = pd.read_csv("QUIMICA.csv")

In [9]:
df.head()

,subject,year,topic,exam,exercise,statement
0,Química,2012,Equilibrio Químico,Junio,"Ejercicio 5, Opción B","PCl;(g) Z2 PCl,(g) + Cl,(g)\nCuando se alcanza..."
1,Química,2012,Equilibrio Químico,Reserva 1,"Ejercicio 6, Opción A",a) El valor de Kc.\nb) El grado de disociación...
2,Química,2012,Equilibrio Químico,Reserva 2,"Ejercicio 3, Opción A",a) Aumentar la temperatura.\nb) Retirar del re...
3,Química,2012,Equilibrio Químico,Reserva 2,"Ejercicio 6, Opción B",temperatura de 11 *C la presión es de 0?3 atm....
4,Química,2012,Equilibrio Químico,Reserva 3,"Ejercicio 3, Opción A","reacción en cada uno de los siguientes casos, ..."


In [11]:
df.subject.unique()[0]

'Química'

In [28]:
def clasificar_ejercicios(dataset: pd.DataFrame) -> list[str]:
    subject = dataset.subject.unique()[0]   # dataset, no df
    unique_topics = dataset.topic.unique()

    for topic in unique_topics:
        system_prompt = pathlib.Path(f"{topic}.md").read_text()

        subset = dataset[dataset.topic == topic]        # iloc no hace falta aquí
        data_csv = subset.to_csv("data_csv.csv", index=False)
        
        data_text = pathlib.Path("data_csv.csv").read_text()        # subset, no df completo

        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=2048,
            system=system_prompt,
            messages=[{"role": "user",
                       "content": f"Here is the data: \n\n{data_text}"
                       }]
        )

    return response.content[0].text.strip().splitlines()

In [42]:
response = clasificar_ejercicios(df[df.topic == "Equilibrio Químico"])[4:]

In [43]:
response

['Química,2012,Equilibrio Químico,Reserva 3,Ejercicio 3 Opción A,principio de LeChatelier',
 'Química,2012,Equilibrio Químico,Septiembre,Ejercicio 3 Opción B,principio de LeChatelier',
 'Química,2012,Equilibrio Químico,Septiembre,Ejercicio 6 Opción B,uso del grado de disociación (grado disociación (alfa) es dato del ejercicio)',
 'Química,2014,Equilibrio Químico,Junio,Ejercicio 3 Opción A,principio de LeChatelier',
 'Química,2014,Equilibrio Químico,Reserva 1,Ejercicio 3 Opción A,principio de LeChatelier',
 'Química,2014,Equilibrio Químico,Reserva 1,Ejercicio 6 Opción B,cálculo de las presiones parciales',
 'Química,2014,Equilibrio Químico,Reserva 2,Ejercicio 5 Opción B,cálculo de las constantes Kc y Kp',
 'Química,2014,Equilibrio Químico,Reserva 3,Ejercicio 6 Opción A,cálculo de las presiones parciales',
 'Química,2014,Equilibrio Químico,Septiembre,Ejercicio 3 Opción B,principio de LeChatelier',
 'Química,2014,Equilibrio Químico,Septiembre,Ejercicio 6 Opción B,equilibrio heterogéneo co

In [44]:
with open("response.csv", "w") as file:
    for line in response:
        file.write(f"{line}\n")

In [45]:
df2 = pd.read_csv("response.csv")

In [46]:
df2

,Química,2012,Equilibrio Químico,Reserva 3,Ejercicio 3 Opción A,principio de LeChatelier
0,Química,2012,Equilibrio Químico,Septiembre,Ejercicio 3 Opción B,principio de LeChatelier
1,Química,2012,Equilibrio Químico,Septiembre,Ejercicio 6 Opción B,uso del grado de disociación (grado disociació...
2,Química,2014,Equilibrio Químico,Junio,Ejercicio 3 Opción A,principio de LeChatelier
3,Química,2014,Equilibrio Químico,Reserva 1,Ejercicio 3 Opción A,principio de LeChatelier
4,Química,2014,Equilibrio Químico,Reserva 1,Ejercicio 6 Opción B,cálculo de las presiones parciales
5,Química,2014,Equilibrio Químico,Reserva 2,Ejercicio 5 Opción B,cálculo de las constantes Kc y Kp
6,Química,2014,Equilibrio Químico,Reserva 3,Ejercicio 6 Opción A,cálculo de las presiones parciales
7,Química,2014,Equilibrio Químico,Septiembre,Ejercicio 3 Opción B,principio de LeChatelier
8,Química,2014,Equilibrio Químico,Septiembre,Ejercicio 6 Opción B,equilibrio heterogéneo con cálculo de la canti...
9,Química,2015,Equilibrio Químico,Junio,Ejercicio 4 Opción A,principio de LeChatelier
